# Regression-based physical copper price and certificate bubble

This notebook is an experimental alternative to ratio interpolation. It estimates IME physical copper price from free-market USD/IRR and LME cash copper USD/kg, compares a linear model with a degree-2 polynomial Ridge model using time-series cross-validation, and calculates the certificate bubble from the selected model. The production pipeline is not modified.

In [ ]:
from bisect import bisect_right
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. Load the four source datasets

In [ ]:
def first_existing(*candidates):
    path = next((Path(candidate) for candidate in candidates if Path(candidate).exists()), None)
    if path is None:
        raise FileNotFoundError(f"None of these paths exists: {candidates}")
    return path


certificate_path = first_existing(
    "../data/raw/certificate/copper_certificate_raw.csv",
    "commodity/copper/data/raw/certificate/copper_certificate_raw.csv",
)
physical_path = first_existing(
    "../data/processed/nci_copper_cash_daily.csv",
    "commodity/copper/data/processed/nci_copper_cash_daily.csv",
)
lme_path = first_existing(
    "../data/raw/lme/copper_lme_raw.csv",
    "commodity/copper/data/raw/lme/copper_lme_raw.csv",
)
usd_path = first_existing(
    "../data/raw/fx/usd_to_rial.csv",
    "commodity/copper/data/raw/fx/usd_to_rial.csv",
)

certificate_raw = pd.read_csv(certificate_path)
physical_daily = pd.read_csv(physical_path)
lme_raw = pd.read_csv(lme_path)
usd_raw = pd.read_csv(usd_path)

{
    "certificate_raw": certificate_raw.shape,
    "physical_daily": physical_daily.shape,
    "lme_raw": lme_raw.shape,
    "usd_raw": usd_raw.shape,
}

## 2. Clean dates and prices

Certificate price is daily VWAP (`TradesValue / TradesVolume`). LME is converted from USD/metric-tonne to USD/kg. Missing weekend or holiday LME/USD observations are joined backward, never from a future date.

In [ ]:
certificate = certificate_raw.copy()
certificate["date"] = pd.to_datetime(certificate["DT"].str[:10])
for column in ["TradesVolume", "TradesValue", "TodaySettlementPrice"]:
    certificate[column] = pd.to_numeric(certificate[column], errors="raise")
certificate = certificate.loc[certificate["TradesVolume"] > 0].copy()
certificate["certificate_price"] = (
    certificate["TradesValue"] / certificate["TradesVolume"]
)
certificate["settlement_check_error"] = (
    certificate["certificate_price"] - certificate["TodaySettlementPrice"]
).abs()
assert certificate["settlement_check_error"].max() <= 0.500001
certificate = certificate[[
    "date", "certificate_price", "TradesVolume", "TradesValue"
]].sort_values("date")

physical = physical_daily.copy()
physical["date"] = pd.to_datetime(physical["physical_trade_date_gregorian"])
physical["physical_price"] = pd.to_numeric(
    physical["physical_weighted_price"], errors="raise"
)
physical = physical[["date", "physical_price", "total_quantity"]].sort_values("date")

lme = lme_raw.loc[lme_raw["cash_settlement"].astype(str).str.strip().ne("-")].copy()
lme["lme_source_date"] = pd.to_datetime(lme["date"])
lme["lme_usd_per_ton"] = pd.to_numeric(
    lme["cash_settlement"].astype(str).str.replace(",", "", regex=False),
    errors="raise",
)
lme["lme_usd_per_kg"] = lme["lme_usd_per_ton"] / 1_000
lme = lme[["lme_source_date", "lme_usd_per_kg"]].sort_values("lme_source_date")

def parse_mixed_gregorian(value):
    parts = [int(part) for part in str(value).replace("-", "/").split("/")]
    if parts[0] >= 1900:
        year, month, day = parts
    else:
        month, day, year = parts
    return pd.Timestamp(year=year, month=month, day=day)

usd = usd_raw.copy()
usd["usd_source_date"] = usd["date_gr"].map(parse_mixed_gregorian)
usd["usd_irr"] = pd.to_numeric(
    usd["price_irr"].astype(str).str.replace(",", "", regex=False),
    errors="raise",
)
usd = usd[["usd_source_date", "usd_irr"]].sort_values("usd_source_date")

assert certificate["date"].is_unique
assert physical["date"].is_unique
assert lme["lme_source_date"].is_unique
assert usd["usd_source_date"].is_unique

## 3. Add backward-looking LME and USD inputs

In [ ]:
def add_market_inputs(frame):
    result = frame.sort_values("date").copy()
    result = pd.merge_asof(
        result,
        lme,
        left_on="date",
        right_on="lme_source_date",
        direction="backward",
    )
    result = pd.merge_asof(
        result.sort_values("date"),
        usd,
        left_on="date",
        right_on="usd_source_date",
        direction="backward",
    )
    result["lme_age_days"] = (result["date"] - result["lme_source_date"]).dt.days
    result["usd_age_days"] = (result["date"] - result["usd_source_date"]).dt.days
    result["intrinsic_price"] = result["lme_usd_per_kg"] * result["usd_irr"]
    if result[["lme_usd_per_kg", "usd_irr"]].isna().any().any():
        raise ValueError("Missing LME or USD inputs after backward merge")
    return result


# Regression training data comes only from the physical market, USD, and LME.
# The certificate dataset is not merged into the regression sample.
REGRESSION_START_DATE = pd.Timestamp("2025-11-12")
physical_regression_sample = physical.loc[
    physical["date"] >= REGRESSION_START_DATE
].copy()
anchors = add_market_inputs(physical_regression_sample)
assert len(anchors) == 25, f"Expected 25 physical observations, found {len(anchors)}"

# Certificate market inputs are prepared separately and are used only after fitting.
certificate_features = add_market_inputs(certificate)

anchors[[
    "date", "physical_price", "usd_irr", "lme_usd_per_kg", "intrinsic_price"
]]

## 4. Compare linear and degree-2 polynomial regression

Only 25 physical-market observations are available, so the polynomial model is regularized with Ridge. `TimeSeriesSplit` preserves temporal order. Model selection uses the lowest out-of-sample RMSE. Certificate prices are not present in `X`, `y`, or the regression sample.

In [ ]:
FEATURES = ["usd_irr", "lme_usd_per_kg"]
TARGET = "physical_price"

X = anchors[FEATURES]
y = anchors[TARGET]

models = {
    "linear": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "polynomial_degree_2_ridge": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-4, 4, 81))),
    ]),
}

tscv = TimeSeriesSplit(n_splits=5)
comparison_rows = []
oos_predictions = {}

for name, model in models.items():
    prediction = pd.Series(np.nan, index=y.index, dtype=float)
    for train_index, test_index in tscv.split(X):
        fold_model = clone(model)
        fold_model.fit(X.iloc[train_index], y.iloc[train_index])
        prediction.iloc[test_index] = fold_model.predict(X.iloc[test_index])
    valid = prediction.notna()
    valid_prediction = prediction.loc[valid]
    actual = y.loc[valid]
    oos_predictions[name] = valid_prediction
    comparison_rows.append({
        "model": name,
        "oos_n": len(actual),
        "MAE": mean_absolute_error(actual, valid_prediction),
        "RMSE": mean_squared_error(actual, valid_prediction) ** 0.5,
        "R2": r2_score(actual, valid_prediction),
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)
model_comparison

## 5. Fit the selected model and inspect anchor fit

In [ ]:
selected_model_name = model_comparison.loc[0, "model"]
selected_model = models[selected_model_name]
selected_model.fit(X, y)

anchors["fitted_physical_price"] = selected_model.predict(X)
anchors["fitted_error_pct"] = (
    anchors["fitted_physical_price"] / anchors["physical_price"] - 1
) * 100

print(f"Selected model: {selected_model_name}")
if selected_model_name == "polynomial_degree_2_ridge":
    print(f"Selected Ridge alpha: {selected_model.named_steps['model'].alpha_}")

anchors[[
    "date", "physical_price", "fitted_physical_price", "fitted_error_pct"
]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(
    anchors["physical_price"],
    anchors["fitted_physical_price"],
    s=55,
    color="#1565C0",
    edgecolor="white",
)
limits = [
    min(anchors["physical_price"].min(), anchors["fitted_physical_price"].min()),
    max(anchors["physical_price"].max(), anchors["fitted_physical_price"].max()),
]
ax.plot(limits, limits, "--", color="#263238", label="Perfect fit")
ax.set(
    title=f"Observed vs fitted IME physical price — {selected_model_name}",
    xlabel="Observed IRR/kg",
    ylabel="Fitted IRR/kg",
)
ax.legend()
fig.tight_layout()
plt.show()

## 6. Estimate physical price for all 159 certificate trading days and calculate bubble

Unlike linear ratio interpolation, regression can produce estimates before the first and after the last physical anchor. These are model estimates—not observed physical trades—and should be interpreted with the cross-validation results.

In [ ]:
bubble_regression = certificate_features.copy()
bubble_regression["estimated_physical_price"] = selected_model.predict(
    bubble_regression[FEATURES]
)
if (bubble_regression["estimated_physical_price"] <= 0).any():
    raise ValueError("The selected regression produced a non-positive physical price")

bubble_regression["certificate_bubble_irr_per_kg"] = (
    bubble_regression["certificate_price"]
    - bubble_regression["estimated_physical_price"]
)
bubble_regression["certificate_bubble_pct"] = (
    bubble_regression["certificate_price"]
    / bubble_regression["estimated_physical_price"]
    - 1
) * 100
bubble_regression["is_physical_anchor_date"] = bubble_regression["date"].isin(anchors["date"])
bubble_regression["regression_model"] = selected_model_name

bubble_regression[[
    "date", "certificate_price", "estimated_physical_price",
    "certificate_bubble_pct", "is_physical_anchor_date"
]].describe(include="all")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    bubble_regression["date"],
    bubble_regression["certificate_bubble_pct"],
    color="#1565C0",
    linewidth=1.8,
    label=f"Regression bubble ({selected_model_name})",
)
anchor_bubbles = bubble_regression.loc[bubble_regression["is_physical_anchor_date"]]
ax.scatter(
    anchor_bubbles["date"],
    anchor_bubbles["certificate_bubble_pct"],
    color="#D32F2F",
    edgecolor="white",
    linewidth=0.7,
    s=45,
    zorder=3,
    label="Physical anchor date",
)
ax.axhline(0, color="#263238", linewidth=1.1, linestyle="--")
ax.fill_between(
    bubble_regression["date"],
    bubble_regression["certificate_bubble_pct"],
    0,
    where=bubble_regression["certificate_bubble_pct"].ge(0),
    color="#43A047",
    alpha=0.12,
    interpolate=True,
)
ax.fill_between(
    bubble_regression["date"],
    bubble_regression["certificate_bubble_pct"],
    0,
    where=bubble_regression["certificate_bubble_pct"].lt(0),
    color="#E53935",
    alpha=0.12,
    interpolate=True,
)
ax.set(
    title="Daily Copper Certificate Bubble — Regression Estimate",
    xlabel="Date",
    ylabel="Bubble (%)",
)
ax.legend(frameon=True)
ax.margins(x=0.01)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 7. Optional: inspect or save the experimental result

The save command is intentionally commented out. Uncomment it only if the regression result is accepted after reviewing cross-validation diagnostics.

In [ ]:
regression_output = bubble_regression[[
    "date",
    "certificate_price",
    "TradesVolume",
    "lme_source_date",
    "lme_age_days",
    "lme_usd_per_kg",
    "usd_source_date",
    "usd_age_days",
    "usd_irr",
    "intrinsic_price",
    "estimated_physical_price",
    "certificate_bubble_irr_per_kg",
    "certificate_bubble_pct",
    "is_physical_anchor_date",
    "regression_model",
]].copy()

regression_output.head()

# Optional save after reviewing model diagnostics:
# output_path = Path("data/processed/copper_certificate_bubble_regression.csv")
# output_path.parent.mkdir(parents=True, exist_ok=True)
# regression_output.to_csv(output_path, index=False, encoding="utf-8-sig")

## 8. Approved interpolated-ratio certificate bubble

This section plots the previously calculated production output in `copper_certificate_bubble.csv`. It is kept separate from the experimental regression bubble above.

In [ ]:
approved_bubble_path = first_existing(
    "../data/processed/copper_certificate_bubble.csv",
    "commodity/copper/data/processed/copper_certificate_bubble.csv",
)
approved_bubble = pd.read_csv(approved_bubble_path, parse_dates=["date"])
approved_bubble["certificate_bubble_pct"] = pd.to_numeric(
    approved_bubble["certificate_bubble_pct"], errors="raise"
)
approved_bubble = approved_bubble.sort_values("date").reset_index(drop=True)
approved_bubble["is_observed_ratio"] = approved_bubble["physical_ratio_method"].eq("observed")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    approved_bubble["date"],
    approved_bubble["certificate_bubble_pct"],
    color="#1565C0",
    linewidth=1.8,
    label="Certificate bubble (interpolated physical ratio)",
)
observed_ratio_days = approved_bubble.loc[approved_bubble["is_observed_ratio"]]
ax.scatter(
    observed_ratio_days["date"],
    observed_ratio_days["certificate_bubble_pct"],
    color="#D32F2F",
    edgecolor="white",
    linewidth=0.7,
    s=45,
    zorder=3,
    label="Observed physical-ratio date",
)
ax.axhline(0, color="#263238", linestyle="--", linewidth=1.1)
ax.set(
    title="Daily Copper Certificate Bubble — Interpolated Physical Ratio",
    xlabel="Date",
    ylabel="Bubble (%)",
)
ax.legend(frameon=True)
ax.margins(x=0.01)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 9. Observed IME physical-market premium over intrinsic copper value

For the 25 actual physical-market observations, intrinsic value is `LME cash USD/kg × free-market USD/IRR`. The plotted premium is `(observed physical price / intrinsic price - 1) × 100`.

In [ ]:
physical_intrinsic_bubble = anchors[[
    "date", "physical_price", "lme_usd_per_kg", "usd_irr", "intrinsic_price"
]].copy()
assert len(physical_intrinsic_bubble) == 25
physical_intrinsic_bubble["physical_vs_intrinsic_bubble_pct"] = (
    physical_intrinsic_bubble["physical_price"]
    / physical_intrinsic_bubble["intrinsic_price"]
    - 1
) * 100

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    physical_intrinsic_bubble["date"],
    physical_intrinsic_bubble["physical_vs_intrinsic_bubble_pct"],
    color="#6A1B9A",
    linewidth=1.5,
    marker="o",
    markersize=5,
    label="Observed IME physical premium",
)
ax.axhline(0, color="#263238", linestyle="--", linewidth=1.1)
ax.set(
    title="Observed IME Physical Copper Price vs Intrinsic LME–FX Value",
    xlabel="Physical trade date",
    ylabel="Premium / discount (%)",
)
ax.legend(frameon=True)
ax.margins(x=0.02)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

physical_intrinsic_bubble

## 10. Certificate premium over intrinsic copper value

This measure compares certificate VWAP directly with `LME cash USD/kg × free-market USD/IRR`; it does not use observed, interpolated, or regression-estimated IME physical price.

In [ ]:
certificate_intrinsic_bubble = certificate_features[[
    "date", "certificate_price", "lme_usd_per_kg", "usd_irr", "intrinsic_price"
]].copy()
certificate_intrinsic_bubble["certificate_vs_intrinsic_bubble_pct"] = (
    certificate_intrinsic_bubble["certificate_price"]
    / certificate_intrinsic_bubble["intrinsic_price"]
    - 1
) * 100

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    certificate_intrinsic_bubble["date"],
    certificate_intrinsic_bubble["certificate_vs_intrinsic_bubble_pct"],
    color="#EF6C00",
    linewidth=1.8,
    label="Certificate premium over intrinsic value",
)
ax.axhline(0, color="#263238", linestyle="--", linewidth=1.1)
ax.set(
    title="Copper Certificate Price vs Intrinsic LME–FX Value",
    xlabel="Date",
    ylabel="Premium / discount (%)",
)
ax.legend(frameon=True)
ax.margins(x=0.01)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

certificate_intrinsic_bubble

## 11. Full-history IME physical copper premium over intrinsic value

This section extends the physical-market analysis to the complete available history. It uses every positive-trade observation in `nci_copper_cash_daily.csv`, not only the recent certificate period. Intrinsic value is `LME cash USD/kg × free-market USD/IRR`.

In [ ]:
physical_full_history = add_market_inputs(physical).copy()
physical_full_history["physical_vs_intrinsic_bubble_pct"] = (
    physical_full_history["physical_price"]
    / physical_full_history["intrinsic_price"]
    - 1
) * 100

assert len(physical_full_history) == len(physical)
assert physical_full_history["physical_vs_intrinsic_bubble_pct"].notna().all()

print(f"Physical observations: {len(physical_full_history):,}")
print(
    f"Coverage: {physical_full_history['date'].min().date()} "
    f"through {physical_full_history['date'].max().date()}"
)
physical_full_history[[
    "date",
    "physical_price",
    "lme_usd_per_kg",
    "usd_irr",
    "intrinsic_price",
    "physical_vs_intrinsic_bubble_pct",
]].describe(include="all")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(
    physical_full_history["date"],
    physical_full_history["physical_vs_intrinsic_bubble_pct"],
    color="#6A1B9A",
    linewidth=1.25,
    marker="o",
    markersize=2.8,
    alpha=0.85,
    label="Observed IME physical premium",
)
ax.axhline(0, color="#263238", linestyle="--", linewidth=1.1)
ax.fill_between(
    physical_full_history["date"],
    physical_full_history["physical_vs_intrinsic_bubble_pct"],
    0,
    where=physical_full_history["physical_vs_intrinsic_bubble_pct"].ge(0),
    color="#43A047",
    alpha=0.10,
    interpolate=True,
)
ax.fill_between(
    physical_full_history["date"],
    physical_full_history["physical_vs_intrinsic_bubble_pct"],
    0,
    where=physical_full_history["physical_vs_intrinsic_bubble_pct"].lt(0),
    color="#E53935",
    alpha=0.10,
    interpolate=True,
)
ax.set(
    title="Full-History IME Physical Copper Price vs Intrinsic LME–FX Value",
    xlabel="Physical trade date",
    ylabel="Premium / discount (%)",
)
ax.legend(frameon=True)
ax.margins(x=0.005)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 12. Corrected regression: physical price on intrinsic LME–FX value

This section supersedes the earlier experimental two-feature regression. The sole explanatory variable is intrinsic copper value: `LME cash USD/kg × free-market USD/IRR`. Certificate price is not used in model fitting. Three specifications are compared: proportional (no intercept), linear with intercept, and degree-2 polynomial Ridge.

In [ ]:
INTRINSIC_FEATURE = ["intrinsic_price"]
X_intrinsic = anchors[INTRINSIC_FEATURE]
y_physical = anchors["physical_price"]

intrinsic_models = {
    "proportional_no_intercept": LinearRegression(fit_intercept=False),
    "linear_with_intercept": LinearRegression(),
    "polynomial_degree_2_ridge": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-4, 4, 81))),
    ]),
}

intrinsic_comparison_rows = []
intrinsic_oos_predictions = {}
intrinsic_tscv = TimeSeriesSplit(n_splits=5)

for name, model in intrinsic_models.items():
    prediction = pd.Series(np.nan, index=y_physical.index, dtype=float)
    for train_index, test_index in intrinsic_tscv.split(X_intrinsic):
        fold_model = clone(model)
        fold_model.fit(X_intrinsic.iloc[train_index], y_physical.iloc[train_index])
        prediction.iloc[test_index] = fold_model.predict(X_intrinsic.iloc[test_index])
    valid = prediction.notna()
    actual = y_physical.loc[valid]
    predicted = prediction.loc[valid]
    intrinsic_oos_predictions[name] = predicted
    intrinsic_comparison_rows.append({
        "model": name,
        "oos_n": len(actual),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    })

intrinsic_model_comparison = (
    pd.DataFrame(intrinsic_comparison_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
intrinsic_model_comparison

In [ ]:
selected_intrinsic_model_name = intrinsic_model_comparison.loc[0, "model"]
selected_intrinsic_model = clone(intrinsic_models[selected_intrinsic_model_name])
selected_intrinsic_model.fit(X_intrinsic, y_physical)

anchors_intrinsic_regression = anchors.copy()
anchors_intrinsic_regression["fitted_physical_price"] = (
    selected_intrinsic_model.predict(anchors_intrinsic_regression[INTRINSIC_FEATURE])
)
anchors_intrinsic_regression["fitted_error_pct"] = (
    anchors_intrinsic_regression["fitted_physical_price"]
    / anchors_intrinsic_regression["physical_price"]
    - 1
) * 100

bubble_intrinsic_regression = certificate_features.copy()
bubble_intrinsic_regression["estimated_physical_price"] = (
    selected_intrinsic_model.predict(bubble_intrinsic_regression[INTRINSIC_FEATURE])
)
if (bubble_intrinsic_regression["estimated_physical_price"] <= 0).any():
    raise ValueError("The selected intrinsic regression produced a non-positive price")
bubble_intrinsic_regression["certificate_bubble_pct"] = (
    bubble_intrinsic_regression["certificate_price"]
    / bubble_intrinsic_regression["estimated_physical_price"]
    - 1
) * 100
bubble_intrinsic_regression["is_physical_anchor_date"] = (
    bubble_intrinsic_regression["date"].isin(anchors["date"])
)

print(f"Selected corrected model: {selected_intrinsic_model_name}")
anchors_intrinsic_regression[[
    "date", "intrinsic_price", "physical_price",
    "fitted_physical_price", "fitted_error_pct"
]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

axes[0].scatter(
    anchors_intrinsic_regression["physical_price"],
    anchors_intrinsic_regression["fitted_physical_price"],
    color="#1565C0",
    edgecolor="white",
    s=55,
)
fit_limits = [
    min(anchors_intrinsic_regression["physical_price"].min(), anchors_intrinsic_regression["fitted_physical_price"].min()),
    max(anchors_intrinsic_regression["physical_price"].max(), anchors_intrinsic_regression["fitted_physical_price"].max()),
]
axes[0].plot(fit_limits, fit_limits, "--", color="#263238")
axes[0].set(
    title=f"Observed vs fitted — {selected_intrinsic_model_name}",
    xlabel="Observed physical price (IRR/kg)",
    ylabel="Fitted physical price (IRR/kg)",
)

axes[1].plot(
    bubble_intrinsic_regression["date"],
    bubble_intrinsic_regression["certificate_bubble_pct"],
    color="#6A1B9A",
    linewidth=1.7,
)
anchor_regression_bubbles = bubble_intrinsic_regression.loc[
    bubble_intrinsic_regression["is_physical_anchor_date"]
]
axes[1].scatter(
    anchor_regression_bubbles["date"],
    anchor_regression_bubbles["certificate_bubble_pct"],
    color="#D32F2F",
    edgecolor="white",
    s=40,
    zorder=3,
)
axes[1].axhline(0, color="#263238", linestyle="--", linewidth=1.0)
axes[1].set(
    title="Certificate bubble — corrected intrinsic regression",
    xlabel="Date",
    ylabel="Bubble (%)",
)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()